In [1]:
import pandas as pd
df = pd.read_csv("data-mlt/MLT.csv")
dupes_mask = df.duplicated(subset=["SMILES", "cell_id"], keep=False)
assert not dupes_mask.any(), "Duplicate (SMILES, cell_id) pairs found!"

In [2]:
import logging
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch_geometric.loader import DataLoader as PyGDataLoader

from mltutils.models import LatentGeneExpressionGNN, GeneExpressionAutoencoder
from mltutils.utils import (
    MoleculeGeneLatentDataset,
    load_gene_expression_data,
    train,               
    evaluate,         
)

logging.getLogger("deepchem").setLevel(logging.WARNING)
cell_line_id = "JURKAT"
CONFIG = {
    "data_folder": Path("data-mlt/"),
    "train_autoencoder": True,
    "train_prediction_model": True,
    "autoencoder_weights_path": f"autoencoder_weights_{cell_line_id}.pth",            
    "prediction_model_weights_path": f"prediction_model_weights_{cell_line_id}.pth",  
    "learning_rate": 3e-4,
    "weight_decay": 1e-6,
    "num_epochs": 100,
    "hidden_dim": 128,
    "cell_line_embedding_dim": 4,
    "batch_size": 32,
    "batch_size_ae": 32,
    "latent_dim": 64,
    "device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    "n_epochs_autoencoder": 50,
    "autoencoder_lr": 1e-3,
    "autoencoder_weight_decay": 1e-6,
    "train_smiles_file": "MLT.csv",
    "seeds": 42,
}
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(message)s")
logger = logging.getLogger(__name__)

def load_data(config):
    logger.info("Loading data...")
    data_path = config["data_folder"] / config["train_smiles_file"]
    return load_gene_expression_data(data_path)

def preprocess_data(expr_train, expr_test):
    logger.info("Normalizing gene expression data...")
    scaler = StandardScaler()
    expr_train_normalized = scaler.fit_transform(expr_train)
    expr_test_normalized  = scaler.transform(expr_test)

    lower = np.percentile(expr_train_normalized, 2)
    upper = np.percentile(expr_train_normalized, 98)
    expr_train_capped = np.clip(expr_train_normalized, lower, upper)
    expr_test_capped  = np.clip(expr_test_normalized,  lower, upper)
    return expr_train_capped, expr_test_capped, scaler


# ================== Autoencoder===================

def train_autoencoder_best_mse(expr_train_normalized, config):

    expr_train_ae, expr_val_ae = train_test_split(
        expr_train_normalized, test_size=0.2, random_state= config["seeds"]
    )
    device = config["device"]

    # Dataloader
    expr_train_tensor = torch.as_tensor(expr_train_ae, dtype=torch.float32)
    expr_val_tensor   = torch.as_tensor(expr_val_ae,   dtype=torch.float32)
    train_loader_ae = DataLoader(expr_train_tensor, batch_size=config["batch_size_ae"], shuffle=True)
    val_loader_ae   = DataLoader(expr_val_tensor,   batch_size=config["batch_size_ae"], shuffle=False)

    # Model/opt/sched
    autoencoder = GeneExpressionAutoencoder(
        input_dim=expr_train_normalized.shape[1],
        latent_dim=config["latent_dim"]
    ).to(device)

    criterion = nn.MSELoss()
    optimizer = optim.Adam(
        autoencoder.parameters(),
        lr=config["autoencoder_lr"],
        weight_decay=config["autoencoder_weight_decay"],
    )
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.2, patience=5
    )

    best_val = float("inf")
    best_state = None
    best_epoch = -1

    logger.info("Training autoencoder (saving best Val MSE)...")
    for epoch in range(config["n_epochs_autoencoder"]):
        # train
        autoencoder.train()
        train_loss_sum, n_train = 0.0, 0
        for batch in train_loader_ae:
            batch = batch.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            recon = autoencoder(batch)              # forward 
            loss  = criterion(recon, batch)
            loss.backward()
            optimizer.step()
            train_loss_sum += loss.item() * batch.size(0)
            n_train += batch.size(0)
        train_loss = train_loss_sum / max(1, n_train)

        # val
        autoencoder.eval()
        val_loss_sum, n_val = 0.0, 0
        with torch.no_grad():
            for batch in val_loader_ae:
                batch = batch.to(device, non_blocking=True)
                recon = autoencoder(batch)
                loss  = criterion(recon, batch)
                val_loss_sum += loss.item() * batch.size(0)
                n_val += batch.size(0)
        val_loss = val_loss_sum / max(1, n_val)
        scheduler.step(val_loss)

        logger.info(f"[AE Epoch {epoch+1}/{config['n_epochs_autoencoder']}] "
                    f"Train MSE={train_loss:.6f} | Val MSE={val_loss:.6f}")

        # save best
        if val_loss < best_val:
            best_val = val_loss
            best_epoch = epoch + 1
            best_state = {k: v.detach().cpu().clone() for k, v in autoencoder.state_dict().items()}
            torch.save(best_state, config["autoencoder_weights_path"])
            logger.info(f" New best AE Val MSE={best_val:.6f} at epoch {best_epoch}. "
                        f"Saved -> {config['autoencoder_weights_path']}")


    if best_state is not None:
        autoencoder.load_state_dict(best_state)
    autoencoder.eval()
    return autoencoder


def train_or_load_autoencoder(expr_train_normalized, config):
    if config["train_autoencoder"]:
        return train_autoencoder_best_mse(expr_train_normalized, config)

    logger.info(f"Loading autoencoder best weights from {config['autoencoder_weights_path']}...")
    autoencoder = GeneExpressionAutoencoder(
        input_dim=expr_train_normalized.shape[1],
        latent_dim=config["latent_dim"]
    ).to(config["device"])
    try:
        autoencoder.load_state_dict(torch.load(
            config["autoencoder_weights_path"],
            map_location=config["device"]
        ))
        autoencoder.eval()
        return autoencoder
    except FileNotFoundError:
        logger.error("Best AE weights not found. Training instead...")
        return train_autoencoder_best_mse(expr_train_normalized, config)



def train_prediction_model_best_r2(
    model,
    train_loader,
    val_loader,
    autoencoder,
    scaler,
    config,
):

    device = config["device"]
    criterion = nn.MSELoss()
    optimizer = optim.Adam(
        model.parameters(),
        lr=config["learning_rate"],
        weight_decay=config["weight_decay"],
    )

    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=config["num_epochs"]
    )

    best_r2 = -float("inf")
    best_state = None
    best_epoch = -1

    for epoch in range(config["num_epochs"]):

        train(
            model=model,
            train_loader=train_loader,
            test_loader=val_loader,    
            criterion=criterion,
            optimizer=optimizer,
            scheduler=scheduler,      
            num_epochs=1,               
            device=device,
            autoencoder=autoencoder,
            scaler=scaler,
        )


        val_loss, val_r2, val_pearson = evaluate(
            model, val_loader, criterion, device, autoencoder, scaler
        )
        logger.info(
            f"[Epoch {epoch+1}/{config['num_epochs']}] "
            f"Val Loss={val_loss:.4f} | Val R²={val_r2:.4f} | Val Pearson={val_pearson:.4f}"
        )

        if val_r2 > best_r2:
            best_r2 = val_r2
            best_epoch = epoch + 1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            torch.save(best_state, config["prediction_model_weights_path"])
            logger.info(f" New best Val R²={best_r2:.4f} at epoch {best_epoch}. "
                        f"Saved -> {config['prediction_model_weights_path']}")

    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    return model, {"Best Val R2": best_r2, "Best Epoch": best_epoch}


def train_or_load_prediction_model(
    train_loader, val_loader, autoencoder, scaler, num_cell_lines, config
):

    num_node_features = train_loader.dataset[0][0].num_node_features

    model = LatentGeneExpressionGNN(
        num_node_features=num_node_features,
        hidden_dim=config["hidden_dim"],
        num_cell_lines=num_cell_lines,
        cell_line_embedding_dim=config["cell_line_embedding_dim"],
        latent_dim=config["latent_dim"],
    ).to(config["device"])

    if config["train_prediction_model"]:
        logger.info("Training prediction model (saving best Val R2)...")
        return train_prediction_model_best_r2(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            autoencoder=autoencoder,
            scaler=scaler,
            config=config,
        )

    logger.info(f"Loading best-R2 weights from {config['prediction_model_weights_path']}...")
    try:
        model.load_state_dict(torch.load(
            config["prediction_model_weights_path"],
            map_location=config["device"]
        ))
        model.eval()
        return model, None
    except FileNotFoundError:
        logger.error("Best-R2 weights not found. Training instead...")
        return train_prediction_model_best_r2(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            autoencoder=autoencoder,
            scaler=scaler,
            config=config,
        )



def encode_data(autoencoder, data_normalized, config):
    logger.info("Encoding data using the trained autoencoder...")
    autoencoder.eval()
    with torch.no_grad():
        encoded = autoencoder.encoder(
            torch.as_tensor(data_normalized, dtype=torch.float32, device=config["device"])
        ).detach().cpu().numpy()
    return encoded



def _pick_col(df: pd.DataFrame, candidates, required=True, default=None):
    for c in candidates:
        if c in df.columns:
            return df[c]
    if required:
        raise KeyError(f"None of columns {candidates} found in file.")
    return pd.Series([default] * len(df), index=df.index)



def main(config):

    config["data_folder"].mkdir(parents=True, exist_ok=True)


    (
        graph_data,
        cell_lines,
        gene_expression,
        num_cell_lines,
        gene_names,
        cell_line_encoder,
    ) = load_data(config)


    (
        graphs_tmp, graphs_test,
        cell_lines_tmp, cell_lines_test,
        expr_tmp, expr_test,
    ) = train_test_split(
        graph_data, cell_lines, gene_expression, test_size=0.2, random_state=config["seeds"]
    )
    (
        graphs_train, graphs_val,
        cell_lines_train, cell_lines_val,
        expr_train, expr_val,
    ) = train_test_split(
        graphs_tmp, cell_lines_tmp, expr_tmp, test_size=0.2, random_state=config["seeds"]
    )


    expr_train_norm, expr_val_norm, scaler = preprocess_data(expr_train, expr_val)
    _, expr_test_norm, _ = preprocess_data(expr_train, expr_test)


    autoencoder = train_or_load_autoencoder(expr_train_norm, config)


    z_train = encode_data(autoencoder, expr_train_norm, config)
    z_val   = encode_data(autoencoder, expr_val_norm,   config)
    z_test  = encode_data(autoencoder, expr_test_norm,  config)


    train_dataset = MoleculeGeneLatentDataset(graphs_train, cell_lines_train, z_train)
    val_dataset   = MoleculeGeneLatentDataset(graphs_val,   cell_lines_val,   z_val)
    test_dataset  = MoleculeGeneLatentDataset(graphs_test,  cell_lines_test,  z_test)

    train_loader = PyGDataLoader(train_dataset, batch_size=config["batch_size"], shuffle=True)
    val_loader   = PyGDataLoader(val_dataset,   batch_size=config["batch_size"])
    test_loader  = PyGDataLoader(test_dataset,  batch_size=config["batch_size"])

    model, best_metrics = train_or_load_prediction_model(
        train_loader, val_loader, autoencoder, scaler, num_cell_lines, config
    )

    criterion = nn.MSELoss()
    test_loss, test_r2, test_pearson = evaluate(
        model, test_loader, criterion, config["device"], autoencoder, scaler
    )
    final_metrics = {
        "Test Loss": test_loss,
        "Test R2": test_r2,
        "Test Pearson": test_pearson,
        **(best_metrics or {}),
    }

    return autoencoder, model, final_metrics

if __name__ == "__main__":
    autoencoder, model, final_metrics = main(CONFIG)
    CONFIG["data_folder"].mkdir(parents=True, exist_ok=True)
    logger.info(f"Final metrics: {final_metrics}")


Instructions for updating:
experimental_relax_shapes is deprecated, use reduce_retracing instead


Skipped loading modules with pytorch-geometric dependency, missing a dependency. No module named 'dgl'
Skipped loading modules with pytorch-lightning dependency, missing a dependency. No module named 'lightning'
Skipped loading some Jax models, missing a dependency. No module named 'jax'
2025-09-13 00:08:04,477 - Loading data...
2025-09-13 00:08:34,617 - Normalizing gene expression data...
2025-09-13 00:08:34,678 - Normalizing gene expression data...
2025-09-13 00:08:34,844 - Training autoencoder (saving best Val MSE)...
2025-09-13 00:08:36,246 - [AE Epoch 1/50] Train MSE=1.717999 | Val MSE=0.753926
2025-09-13 00:08:36,254 -  New best AE Val MSE=0.753926 at epoch 1. Saved -> autoencoder_weights_JURKAT.pth
2025-09-13 00:08:36,343 - [AE Epoch 2/50] Train MSE=0.671002 | Val MSE=0.401484
2025-09-13 00:08:36,349 -  New best AE Val MSE=0.401484 at epoch 2. Saved -> autoencoder_weights_JURKAT.pth
2025-09-13 00:08:36,441 - [AE Epoch 3/50] Train MSE=0.434981 | Val MSE=0.297987
2025-09-13 00:08: